# Exploratory Time Series Analysis

## Comprehensive Time Series Analysis Toolkit

This notebook provides a complete framework for analyzing time series data, including:
- Stationarity testing
- Seasonality detection
- Decomposition analysis
- Outlier detection
- Autocorrelation analysis
- Feature engineering

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
from typing import Dict, Any, List, Tuple, Optional
import warnings

warnings.filterwarnings("ignore")

# Statistical analysis
from scipy import stats, signal
from statsmodels.tsa.seasonal import seasonal_decompose, STL
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Visualization
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

# Configure plotting
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

## 1. Time Series Analyzer Class

In [ ]:
class TimeSeriesAnalyzer:
    """Comprehensive time series analysis toolkit."""

    def __init__(
        self, data: pd.DataFrame, date_col: str, target_col: str, freq: str = None
    ):
        """Initialize the analyzer.

        Parameters:
        -----------
        data : pd.DataFrame
            Input dataframe
        date_col : str
            Name of the date column
        target_col : str
            Name of the target variable column
        freq : str, optional
            Frequency of the time series (e.g., 'D', 'M', 'H')
        """
        self.data = data.copy()
        self.date_col = date_col
        self.target_col = target_col

        # Convert to datetime and set index
        self.data[date_col] = pd.to_datetime(self.data[date_col])
        self.data = self.data.set_index(date_col).sort_index()

        # Infer frequency if not provided
        if freq is None:
            self.freq = pd.infer_freq(self.data.index)
        else:
            self.freq = freq

        # Store the series
        self.series = self.data[target_col].dropna()

    def perform_eda(self) -> Dict[str, Any]:
        """Perform comprehensive exploratory data analysis."""
        print("Performing Exploratory Data Analysis...")

        results = {
            "basic_stats": self._basic_statistics(),
            "missing_data": self._analyze_missing_data(),
            "stationarity": self._test_stationarity(),
            "seasonality": self._detect_seasonality(),
            "decomposition": self._decompose_series(),
            "autocorrelation": self._analyze_autocorrelation(),
            "outliers": self._detect_outliers(),
            "distribution": self._analyze_distribution(),
        }

        return results

    def _basic_statistics(self) -> Dict[str, Any]:
        """Calculate basic statistics."""
        series = self.series

        return {
            "count": len(series),
            "mean": series.mean(),
            "std": series.std(),
            "min": series.min(),
            "max": series.max(),
            "median": series.median(),
            "skewness": series.skew(),
            "kurtosis": series.kurtosis(),
            "cv": series.std() / series.mean(),  # Coefficient of variation
            "date_range": (series.index.min(), series.index.max()),
            "frequency": self.freq,
        }

    def _analyze_missing_data(self) -> Dict[str, Any]:
        """Analyze missing data patterns."""
        missing_count = self.data[self.target_col].isna().sum()
        missing_pct = missing_count / len(self.data) * 100

        # Find consecutive missing periods
        is_missing = self.data[self.target_col].isna()
        missing_groups = is_missing.ne(is_missing.shift()).cumsum()[is_missing]

        consecutive_missing = []
        for group in missing_groups.unique():
            group_data = missing_groups[missing_groups == group]
            consecutive_missing.append(
                {
                    "start": group_data.index[0],
                    "end": group_data.index[-1],
                    "length": len(group_data),
                }
            )

        return {
            "missing_count": missing_count,
            "missing_percentage": missing_pct,
            "consecutive_missing_periods": consecutive_missing,
        }

    def _test_stationarity(self) -> Dict[str, Any]:
        """Test for stationarity using multiple methods."""
        series = self.series

        # Augmented Dickey-Fuller test
        adf_result = adfuller(series, autolag="AIC")

        # KPSS test
        kpss_result = kpss(series, regression="c", nlags="auto")

        # Rolling statistics
        window = min(12, len(series) // 4)
        rolling_mean = series.rolling(window=window).mean()
        rolling_std = series.rolling(window=window).std()

        # Check if variance is constant
        first_half_std = series[: len(series) // 2].std()
        second_half_std = series[len(series) // 2 :].std()
        variance_ratio = max(first_half_std, second_half_std) / min(
            first_half_std, second_half_std
        )

        return {
            "adf_statistic": adf_result[0],
            "adf_pvalue": adf_result[1],
            "adf_critical_values": adf_result[4],
            "adf_conclusion": "Stationary"
            if adf_result[1] < 0.05
            else "Non-stationary",
            "kpss_statistic": kpss_result[0],
            "kpss_pvalue": kpss_result[1],
            "kpss_critical_values": kpss_result[3],
            "kpss_conclusion": "Stationary"
            if kpss_result[1] > 0.05
            else "Non-stationary",
            "variance_ratio": variance_ratio,
            "is_stationary": adf_result[1] < 0.05 and kpss_result[1] > 0.05,
            "rolling_mean": rolling_mean,
            "rolling_std": rolling_std,
        }

    def _detect_seasonality(self) -> Dict[str, Any]:
        """Detect seasonality patterns using multiple methods."""
        series = self.series

        # Fourier Transform for frequency detection
        fft = np.fft.fft(series.values)
        frequencies = np.fft.fftfreq(len(series))

        # Find dominant frequencies
        power = np.abs(fft) ** 2
        threshold = np.percentile(power, 95)
        dominant_freq_idx = np.where(power[: len(power) // 2] > threshold)[0]

        # Convert frequencies to periods
        dominant_periods = []
        for idx in dominant_freq_idx:
            if frequencies[idx] > 0:
                period = 1 / frequencies[idx]
                dominant_periods.append(abs(period))

        # Seasonal decomposition
        decomposition = None
        seasonal_strength = 0

        if len(series) >= 2 * 12:  # Need at least 2 cycles
            try:
                # Try STL decomposition first (more robust)
                stl = STL(series, seasonal=13)  # Use odd number
                decomposition = stl.fit()
                seasonal_strength = 1 - np.var(decomposition.resid) / np.var(series)
            except:
                try:
                    # Fallback to classical decomposition
                    decomposition = seasonal_decompose(
                        series, model="additive", period=12
                    )
                    seasonal_strength = 1 - np.var(
                        decomposition.resid.dropna()
                    ) / np.var(series)
                except:
                    pass

        # Autocorrelation at seasonal lags
        seasonal_acf = {}
        for lag in [7, 12, 30, 365]:
            if lag < len(series) // 2:
                acf_value = acf(series, nlags=lag)[-1]
                seasonal_acf[f"lag_{lag}"] = acf_value

        return {
            "dominant_periods": dominant_periods[:5],  # Top 5 periods
            "seasonal_strength": seasonal_strength,
            "has_strong_seasonality": seasonal_strength > 0.3,
            "seasonal_acf": seasonal_acf,
            "decomposition": decomposition,
        }

    def _decompose_series(self) -> Optional[Any]:
        """Perform time series decomposition."""
        series = self.series

        if len(series) < 24:
            return None

        try:
            # Use STL decomposition
            stl = STL(series, seasonal=13, trend=None)
            decomposition = stl.fit()

            return {
                "trend": decomposition.trend,
                "seasonal": decomposition.seasonal,
                "residual": decomposition.resid,
                "method": "STL",
            }
        except:
            return None

    def _analyze_autocorrelation(self) -> Dict[str, Any]:
        """Analyze autocorrelation and partial autocorrelation."""
        series = self.series

        # Calculate ACF and PACF
        max_lag = min(40, len(series) // 3)
        acf_values = acf(series, nlags=max_lag)
        pacf_values = pacf(series, nlags=max_lag)

        # Find significant lags (outside 95% confidence interval)
        confidence_interval = 1.96 / np.sqrt(len(series))
        significant_acf_lags = (
            np.where(np.abs(acf_values[1:]) > confidence_interval)[0] + 1
        )
        significant_pacf_lags = (
            np.where(np.abs(pacf_values[1:]) > confidence_interval)[0] + 1
        )

        # Ljung-Box test for autocorrelation
        lb_test = acorr_ljungbox(series, lags=min(10, len(series) // 5), return_df=True)

        return {
            "acf_values": acf_values,
            "pacf_values": pacf_values,
            "significant_acf_lags": significant_acf_lags.tolist(),
            "significant_pacf_lags": significant_pacf_lags.tolist(),
            "ljung_box_statistics": lb_test["lb_stat"].values,
            "ljung_box_pvalues": lb_test["lb_pvalue"].values,
            "has_autocorrelation": any(lb_test["lb_pvalue"] < 0.05),
        }

    def _detect_outliers(self) -> Dict[str, Any]:
        """Detect outliers using multiple methods."""
        series = self.series

        # Method 1: IQR
        Q1 = series.quantile(0.25)
        Q3 = series.quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        iqr_outliers = series[(series < lower_bound) | (series > upper_bound)]

        # Method 2: Z-score
        z_scores = np.abs((series - series.mean()) / series.std())
        z_outliers = series[z_scores > 3]

        # Method 3: Isolation Forest
        from sklearn.ensemble import IsolationForest

        iso_forest = IsolationForest(contamination=0.1, random_state=42)
        outlier_labels = iso_forest.fit_predict(series.values.reshape(-1, 1))
        iso_outliers = series[outlier_labels == -1]

        # Method 4: Local Outlier Factor
        from sklearn.neighbors import LocalOutlierFactor

        lof = LocalOutlierFactor(n_neighbors=20, contamination=0.1)
        outlier_labels = lof.fit_predict(series.values.reshape(-1, 1))
        lof_outliers = series[outlier_labels == -1]

        return {
            "iqr_outliers": {
                "count": len(iqr_outliers),
                "percentage": len(iqr_outliers) / len(series) * 100,
                "indices": iqr_outliers.index.tolist(),
                "values": iqr_outliers.values.tolist(),
            },
            "z_score_outliers": {
                "count": len(z_outliers),
                "percentage": len(z_outliers) / len(series) * 100,
                "indices": z_outliers.index.tolist(),
                "values": z_outliers.values.tolist(),
            },
            "isolation_forest_outliers": {
                "count": len(iso_outliers),
                "percentage": len(iso_outliers) / len(series) * 100,
            },
            "lof_outliers": {
                "count": len(lof_outliers),
                "percentage": len(lof_outliers) / len(series) * 100,
            },
        }

    def _analyze_distribution(self) -> Dict[str, Any]:
        """Analyze the distribution of the time series."""
        series = self.series

        # Normality tests
        shapiro_stat, shapiro_p = stats.shapiro(
            series[:5000]
        )  # Shapiro test has sample size limit
        ks_stat, ks_p = stats.kstest(series, "norm", args=(series.mean(), series.std()))

        # Fit different distributions
        distributions = ["norm", "lognorm", "gamma", "expon"]
        best_dist = None
        best_aic = np.inf

        for dist_name in distributions:
            try:
                dist = getattr(stats, dist_name)
                params = dist.fit(series)

                # Calculate AIC
                log_likelihood = np.sum(dist.logpdf(series, *params))
                k = len(params)
                aic = 2 * k - 2 * log_likelihood

                if aic < best_aic:
                    best_aic = aic
                    best_dist = dist_name
            except:
                continue

        return {
            "shapiro_statistic": shapiro_stat,
            "shapiro_pvalue": shapiro_p,
            "is_normal_shapiro": shapiro_p > 0.05,
            "ks_statistic": ks_stat,
            "ks_pvalue": ks_p,
            "is_normal_ks": ks_p > 0.05,
            "best_fit_distribution": best_dist,
            "best_aic": best_aic,
        }

## 2. Visualization Functions

In [ ]:
class TimeSeriesVisualizer:
    """Visualization tools for time series analysis."""

    @staticmethod
    def plot_time_series(series: pd.Series, title: str = "Time Series") -> go.Figure:
        """Plot the time series with interactive features."""
        fig = go.Figure()

        fig.add_trace(
            go.Scatter(
                x=series.index,
                y=series.values,
                mode="lines",
                name="Time Series",
                line=dict(color="blue", width=1),
            )
        )

        # Add range slider
        fig.update_xaxes(rangeslider_visible=True)

        fig.update_layout(
            title=title,
            xaxis_title="Date",
            yaxis_title="Value",
            hovermode="x unified",
            height=500,
        )

        return fig

    @staticmethod
    def plot_decomposition(
        decomposition: Any, title: str = "Time Series Decomposition"
    ) -> go.Figure:
        """Plot time series decomposition."""
        fig = make_subplots(
            rows=4,
            cols=1,
            subplot_titles=("Original", "Trend", "Seasonal", "Residual"),
            vertical_spacing=0.05,
            shared_xaxes=True,
        )

        # Original series
        if hasattr(decomposition, "observed"):
            observed = decomposition.observed
        else:
            observed = (
                decomposition.trend + decomposition.seasonal + decomposition.resid
            )

        fig.add_trace(
            go.Scatter(
                x=observed.index, y=observed.values, mode="lines", name="Original"
            ),
            row=1,
            col=1,
        )

        # Trend
        fig.add_trace(
            go.Scatter(
                x=decomposition.trend.index,
                y=decomposition.trend.values,
                mode="lines",
                name="Trend",
                line=dict(color="red"),
            ),
            row=2,
            col=1,
        )

        # Seasonal
        fig.add_trace(
            go.Scatter(
                x=decomposition.seasonal.index,
                y=decomposition.seasonal.values,
                mode="lines",
                name="Seasonal",
                line=dict(color="green"),
            ),
            row=3,
            col=1,
        )

        # Residual
        fig.add_trace(
            go.Scatter(
                x=decomposition.resid.index,
                y=decomposition.resid.values,
                mode="lines",
                name="Residual",
                line=dict(color="purple"),
            ),
            row=4,
            col=1,
        )

        fig.update_layout(height=800, title_text=title, showlegend=False)
        fig.update_xaxes(title_text="Date", row=4, col=1)

        return fig

    @staticmethod
    def plot_acf_pacf(acf_values: np.ndarray, pacf_values: np.ndarray) -> go.Figure:
        """Plot ACF and PACF."""
        fig = make_subplots(
            rows=2,
            cols=1,
            subplot_titles=(
                "Autocorrelation Function (ACF)",
                "Partial Autocorrelation Function (PACF)",
            ),
            vertical_spacing=0.15,
        )

        n_lags = len(acf_values)
        confidence_interval = 1.96 / np.sqrt(n_lags)

        # ACF
        fig.add_trace(
            go.Bar(
                x=list(range(n_lags)), y=acf_values, name="ACF", marker_color="blue"
            ),
            row=1,
            col=1,
        )

        # Add confidence bands for ACF
        fig.add_hline(
            y=confidence_interval, line_dash="dash", line_color="red", row=1, col=1
        )
        fig.add_hline(
            y=-confidence_interval, line_dash="dash", line_color="red", row=1, col=1
        )

        # PACF
        fig.add_trace(
            go.Bar(
                x=list(range(len(pacf_values))),
                y=pacf_values,
                name="PACF",
                marker_color="green",
            ),
            row=2,
            col=1,
        )

        # Add confidence bands for PACF
        fig.add_hline(
            y=confidence_interval, line_dash="dash", line_color="red", row=2, col=1
        )
        fig.add_hline(
            y=-confidence_interval, line_dash="dash", line_color="red", row=2, col=1
        )

        fig.update_xaxes(title_text="Lag", row=2, col=1)
        fig.update_yaxes(title_text="Correlation", row=1, col=1)
        fig.update_yaxes(title_text="Correlation", row=2, col=1)

        fig.update_layout(height=600, showlegend=False)

        return fig

    @staticmethod
    def plot_stationarity_test(
        series: pd.Series, rolling_mean: pd.Series, rolling_std: pd.Series
    ) -> go.Figure:
        """Plot stationarity test results."""
        fig = make_subplots(
            rows=2,
            cols=1,
            subplot_titles=(
                "Original Series with Rolling Mean",
                "Rolling Standard Deviation",
            ),
            vertical_spacing=0.1,
            shared_xaxes=True,
        )

        # Original series and rolling mean
        fig.add_trace(
            go.Scatter(
                x=series.index,
                y=series.values,
                mode="lines",
                name="Original",
                opacity=0.5,
            ),
            row=1,
            col=1,
        )

        fig.add_trace(
            go.Scatter(
                x=rolling_mean.index,
                y=rolling_mean.values,
                mode="lines",
                name="Rolling Mean",
                line=dict(color="red", width=2),
            ),
            row=1,
            col=1,
        )

        # Rolling std
        fig.add_trace(
            go.Scatter(
                x=rolling_std.index,
                y=rolling_std.values,
                mode="lines",
                name="Rolling Std",
                line=dict(color="green", width=2),
            ),
            row=2,
            col=1,
        )

        fig.update_xaxes(title_text="Date", row=2, col=1)
        fig.update_yaxes(title_text="Value", row=1, col=1)
        fig.update_yaxes(title_text="Standard Deviation", row=2, col=1)

        fig.update_layout(height=600, title_text="Stationarity Analysis")

        return fig

## 3. Load and Analyze Sample Data

In [ ]:
# Generate sample time series data for demonstration
np.random.seed(42)

# Create date range
dates = pd.date_range(start="2020-01-01", end="2023-12-31", freq="D")

# Generate time series with trend, seasonality, and noise
trend = np.linspace(100, 150, len(dates))
seasonal = 10 * np.sin(2 * np.pi * np.arange(len(dates)) / 365.25)  # Annual seasonality
weekly = 5 * np.sin(2 * np.pi * np.arange(len(dates)) / 7)  # Weekly seasonality
noise = np.random.normal(0, 5, len(dates))

# Combine components
values = trend + seasonal + weekly + noise

# Add some outliers
outlier_indices = np.random.choice(len(dates), size=20, replace=False)
values[outlier_indices] += np.random.normal(0, 50, 20)

# Create DataFrame
df = pd.DataFrame({"date": dates, "value": values})

# Add some missing values
missing_indices = np.random.choice(len(df), size=30, replace=False)
df.loc[missing_indices, "value"] = np.nan

print(f"Created time series with {len(df)} observations")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Missing values: {df['value'].isna().sum()}")

df.head()

## 4. Perform Comprehensive Analysis

In [ ]:
# Initialize analyzer
analyzer = TimeSeriesAnalyzer(df, "date", "value")

# Perform comprehensive EDA
results = analyzer.perform_eda()

print("\n" + "=" * 50)
print("EXPLORATORY DATA ANALYSIS RESULTS")
print("=" * 50)

In [ ]:
# Display basic statistics
print("\n📊 BASIC STATISTICS:")
print("-" * 30)
for key, value in results["basic_stats"].items():
    if key != "date_range":
        print(
            f"{key.capitalize()}: {value:.2f}"
            if isinstance(value, (int, float))
            else f"{key.capitalize()}: {value}"
        )
    else:
        print(f"Date Range: {value[0]} to {value[1]}")

In [ ]:
# Display stationarity results
print("\n📈 STATIONARITY TESTS:")
print("-" * 30)
stat_results = results["stationarity"]
print("ADF Test:")
print(f"  - Statistic: {stat_results['adf_statistic']:.4f}")
print(f"  - P-value: {stat_results['adf_pvalue']:.4f}")
print(f"  - Conclusion: {stat_results['adf_conclusion']}")
print("\nKPSS Test:")
print(f"  - Statistic: {stat_results['kpss_statistic']:.4f}")
print(f"  - P-value: {stat_results['kpss_pvalue']:.4f}")
print(f"  - Conclusion: {stat_results['kpss_conclusion']}")
print(
    f"\nOverall: {'✅ Stationary' if stat_results['is_stationary'] else '❌ Non-stationary'}"
)

In [ ]:
# Display seasonality results
print("\n🔄 SEASONALITY DETECTION:")
print("-" * 30)
season_results = results["seasonality"]
print(f"Seasonal Strength: {season_results['seasonal_strength']:.3f}")
print(
    f"Has Strong Seasonality: {'✅ Yes' if season_results['has_strong_seasonality'] else '❌ No'}"
)
if season_results["dominant_periods"]:
    print(
        f"Dominant Periods: {[f'{p:.1f}' for p in season_results['dominant_periods'][:3]]}"
    )

In [ ]:
# Display outlier results
print("\n⚠️ OUTLIER DETECTION:")
print("-" * 30)
outlier_results = results["outliers"]
print(
    f"IQR Method: {outlier_results['iqr_outliers']['count']} outliers ({outlier_results['iqr_outliers']['percentage']:.1f}%)"
)
print(
    f"Z-Score Method: {outlier_results['z_score_outliers']['count']} outliers ({outlier_results['z_score_outliers']['percentage']:.1f}%)"
)
print(
    f"Isolation Forest: {outlier_results['isolation_forest_outliers']['count']} outliers ({outlier_results['isolation_forest_outliers']['percentage']:.1f}%)"
)
print(
    f"Local Outlier Factor: {outlier_results['lof_outliers']['count']} outliers ({outlier_results['lof_outliers']['percentage']:.1f}%)"
)

## 5. Visualizations

In [ ]:
# Initialize visualizer
visualizer = TimeSeriesVisualizer()

# Plot original time series
fig = visualizer.plot_time_series(analyzer.series, "Original Time Series")
fig.show()

In [ ]:
# Plot decomposition if available
if results["seasonality"]["decomposition"] is not None:
    fig = visualizer.plot_decomposition(results["seasonality"]["decomposition"])
    fig.show()

In [ ]:
# Plot ACF and PACF
acf_results = results["autocorrelation"]
fig = visualizer.plot_acf_pacf(acf_results["acf_values"], acf_results["pacf_values"])
fig.show()

In [ ]:
# Plot stationarity analysis
stat_results = results["stationarity"]
fig = visualizer.plot_stationarity_test(
    analyzer.series, stat_results["rolling_mean"], stat_results["rolling_std"]
)
fig.show()

## 6. Distribution Analysis

In [ ]:
# Create distribution plots
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Distribution of Values", "Q-Q Plot"),
    column_widths=[0.5, 0.5],
)

# Histogram
fig.add_trace(
    go.Histogram(x=analyzer.series.values, nbinsx=50, name="Distribution"), row=1, col=1
)

# Q-Q plot
from scipy import stats

theoretical_quantiles = stats.norm.ppf(np.linspace(0.01, 0.99, len(analyzer.series)))
sample_quantiles = np.sort(analyzer.series.values)

fig.add_trace(
    go.Scatter(
        x=theoretical_quantiles,
        y=sample_quantiles,
        mode="markers",
        name="Q-Q Plot",
        marker=dict(size=3),
    ),
    row=1,
    col=2,
)

# Add diagonal line for Q-Q plot
min_val = min(theoretical_quantiles.min(), sample_quantiles.min())
max_val = max(theoretical_quantiles.max(), sample_quantiles.max())
fig.add_trace(
    go.Scatter(
        x=[min_val, max_val],
        y=[min_val, max_val],
        mode="lines",
        line=dict(color="red", dash="dash"),
        name="Normal Line",
    ),
    row=1,
    col=2,
)

fig.update_xaxes(title_text="Value", row=1, col=1)
fig.update_xaxes(title_text="Theoretical Quantiles", row=1, col=2)
fig.update_yaxes(title_text="Frequency", row=1, col=1)
fig.update_yaxes(title_text="Sample Quantiles", row=1, col=2)

fig.update_layout(height=400, title_text="Distribution Analysis", showlegend=False)
fig.show()

# Print distribution test results
dist_results = results["distribution"]
print("\n📊 DISTRIBUTION ANALYSIS:")
print("-" * 30)
print(
    f"Shapiro-Wilk Test: p-value = {dist_results['shapiro_pvalue']:.4f} ({'Normal' if dist_results['is_normal_shapiro'] else 'Not Normal'})"
)
print(
    f"Kolmogorov-Smirnov Test: p-value = {dist_results['ks_pvalue']:.4f} ({'Normal' if dist_results['is_normal_ks'] else 'Not Normal'})"
)
print(f"Best Fit Distribution: {dist_results['best_fit_distribution']}")

## 7. Export Analysis Results

In [ ]:
# Create a comprehensive report
import json


def create_analysis_report(results: Dict[str, Any]) -> Dict[str, Any]:
    """Create a structured analysis report."""

    report = {
        "summary": {
            "n_observations": results["basic_stats"]["count"],
            "date_range": results["basic_stats"]["date_range"],
            "frequency": results["basic_stats"]["frequency"],
            "missing_data_pct": results["missing_data"]["missing_percentage"],
            "is_stationary": results["stationarity"]["is_stationary"],
            "has_seasonality": results["seasonality"]["has_strong_seasonality"],
            "has_autocorrelation": results["autocorrelation"]["has_autocorrelation"],
        },
        "recommendations": [],
    }

    # Add recommendations based on analysis
    if not results["stationarity"]["is_stationary"]:
        report["recommendations"].append(
            "Consider differencing or detrending to achieve stationarity"
        )

    if results["seasonality"]["has_strong_seasonality"]:
        report["recommendations"].append(
            "Include seasonal components in forecasting models (SARIMA, Prophet)"
        )

    if results["missing_data"]["missing_percentage"] > 5:
        report["recommendations"].append(
            "Implement appropriate imputation strategy for missing values"
        )

    if results["outliers"]["iqr_outliers"]["percentage"] > 5:
        report["recommendations"].append("Consider outlier treatment before modeling")

    if results["autocorrelation"]["has_autocorrelation"]:
        report["recommendations"].append(
            "Use models that capture autocorrelation (ARIMA, LSTM)"
        )

    return report


# Generate report
report = create_analysis_report(results)

print("\n" + "=" * 50)
print("ANALYSIS SUMMARY & RECOMMENDATIONS")
print("=" * 50)

print("\n📋 SUMMARY:")
for key, value in report["summary"].items():
    print(f"  - {key.replace('_', ' ').title()}: {value}")

print("\n💡 RECOMMENDATIONS:")
for i, rec in enumerate(report["recommendations"], 1):
    print(f"  {i}. {rec}")

# Save report to JSON
with open("time_series_analysis_report.json", "w") as f:
    json.dump(report, f, indent=2, default=str)

print("\n✅ Analysis report saved to 'time_series_analysis_report.json'")